# LESION N01 — Automatic ISIC 2019 Acquisition, Identity Audit and Grouped Protocol

**GitHub / Kaggle name:** `lesion-n01-dataset-audit-grouped-splits`

This notebook downloads the official ISIC 2019 training image ZIP (about 9.1 GB), ground truth,
and metadata itself. Enable **Internet** in the Kaggle notebook settings. Do not attach a dataset.

The ZIP is processed one image at a time, without expanding all full-resolution originals.
Permanent Kaggle output contains **all** successfully decoded images at 224×224, plus original
checksums, dimensions, labels, provenance, identity audit, group-disjoint splits, and nested
5/10/20/50/100% label-budget manifests. The temporary source ZIP is deleted only on success.

This is an *identity-aware* protocol, **not** a guarantee of patient independence for entries
missing patient IDs. Unverifiable identities are quarantined from the strict primary study.
Near-duplicate similarities are REVIEW FLAGS, never automatically merged as lesion identity.
Groups involved in unresolved candidate links are restricted to TRAIN to protect held-out splits.
ISIC 2019 training imagery is CC BY-NC; comply with the challenge's citation and license rules.
Source: https://challenge.isic-archive.com/data/

In [1]:
import os,sys,re,json,math,time,random,shutil,hashlib,zipfile,platform,subprocess
from pathlib import Path
from collections import defaultdict
from datetime import datetime,timezone
from urllib.parse import urlparse,unquote
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import requests
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image,ImageOps,ImageFile
from scipy.fft import dctn
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm
from IPython.display import display,Image as NotebookImage

ImageFile.LOAD_TRUNCATED_IMAGES=False
SEED=int(os.getenv('N01_SEED','42'))
EXPECTED=int(os.getenv('N01_EXPECTED_IMAGES','25331'))
SIZE=int(os.getenv('N01_IMAGE_SIZE','224'))
JPEG_QUALITY=int(os.getenv('N01_JPEG_QUALITY','94'))
NEAR_PHASH_RADIUS=int(os.getenv('N01_PHASH_RADIUS','6'))
NEAR_DHASH_RADIUS=int(os.getenv('N01_DHASH_RADIUS','8'))
MAX_NEAR_PAIRS=int(os.getenv('N01_MAX_NEAR_PAIRS','150000'))
PIPELINE_VERSION='3.1.0'
SCAN_VERSION='1.0.0'
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
CLASS_TO_ID={c:i for i,c in enumerate(CLASSES)}
ROOT=Path(os.getenv('N01_OUTPUT_DIR','/kaggle/working/lesion-n01-dataset-audit-grouped-splits' if Path('/kaggle/working').exists() else './lesion-n01-dataset-audit-grouped-splits')).resolve()
CACHE=Path(os.getenv('N01_CACHE_DIR','/kaggle/temp/lesion-n01-cache' if Path('/kaggle/temp').exists() else './.lesion-n01-cache')).resolve()
SOURCE_BASE='https://isic-archive.s3.amazonaws.com/challenges/2019/'
URLS={
 'images':os.getenv('N01_IMAGES_URL',SOURCE_BASE+'ISIC_2019_Training_Input.zip'),
 'ground_truth':os.getenv('N01_GT_URL',SOURCE_BASE+'ISIC_2019_Training_GroundTruth.csv'),
 'metadata':os.getenv('N01_META_URL',SOURCE_BASE+'ISIC_2019_Training_Metadata.csv')
}
for sub in ['images','source','audit','splits','budgets','figures','reports']:(ROOT/sub).mkdir(parents=True,exist_ok=True)
CACHE.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED)
print('Notebook: lesion-n01-dataset-audit-grouped-splits.ipynb')
print('Output:',ROOT,'\nTemporary cache:',CACHE)
print('Download policy: official source URLs by default; original ZIP removed after verified completion')
print('Python:',sys.version.split()[0],'Pillow:',Image.__version__,'pandas:',pd.__version__)

Notebook: lesion-n01-dataset-audit-grouped-splits.ipynb
Output: /kaggle/working/lesion-n01-dataset-audit-grouped-splits 
Temporary cache: /kaggle/working/.lesion-n01-cache
Download policy: official source URLs by default; original ZIP removed after verified completion
Python: 3.13.18 Pillow: 13.0.0 pandas: 2.3.3


## 1. Download automatically — resuming incomplete downloads where the server permits

The three official URLs are provided above. For a real Kaggle run **do not edit them**.
The `N01_*_URL` overrides exist only for auditable mirrors and offline synthetic testing;
source overrides are recorded in `reports/protocol.json`. The notebook never searches Kaggle
inputs for images. A request that requires login or lacks Internet access fails visibly.

In [2]:
def file_sha256(path,block=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(block),b''):h.update(chunk)
    return h.hexdigest()

def get_remote_size(session,url):
    if urlparse(url).scheme in ('','file'):
        p=Path(unquote(urlparse(url).path) if urlparse(url).scheme=='file' else url)
        return p.stat().st_size
    try:
        response=session.head(url,allow_redirects=True,timeout=(20,50))
        response.raise_for_status()
        return int(response.headers.get('Content-Length') or 0)
    except requests.RequestException:return 0

def download(session,url,dest,retries=5):
    dest=Path(dest)
    if urlparse(url).scheme in ('','file'):
        src=Path(unquote(urlparse(url).path) if urlparse(url).scheme=='file' else url).resolve()
        if not src.is_file():raise FileNotFoundError(src)
        if src!=dest.resolve():shutil.copyfile(src,dest)
        return dest
    expected=get_remote_size(session,url)
    if dest.exists() and expected and dest.stat().st_size==expected:
        print('Cached complete:',dest.name,f'({expected/2**30:.2f} GiB)')
        return dest
    part=dest.with_name(dest.name+'.part')
    if dest.exists() and (not expected or dest.stat().st_size!=expected):
        if not part.exists():dest.rename(part)
        else:dest.unlink()
    for attempt in range(retries):
        current=part.stat().st_size if part.exists() else 0
        headers={'Range':f'bytes={current}-'} if current else {}
        try:
            with session.get(url,headers=headers,stream=True,timeout=(30,150)) as response:
                if current and response.status_code==416 and expected and current==expected:
                    part.replace(dest);return dest
                response.raise_for_status()
                append=bool(current and response.status_code==206 and response.headers.get('Content-Range','').startswith(f'bytes {current}-'))
                if current and not append:
                    print('Server did not honor resume; restarting:',dest.name)
                    current=0
                total=expected or int(response.headers.get('Content-Length') or 0)+(current if append else 0)
                with open(part,'ab' if append else 'wb') as f,tqdm(total=total,initial=current if append else 0,unit='B',unit_scale=True,desc=dest.name) as bar:
                    for chunk in response.iter_content(chunk_size=4*1024*1024):
                        if chunk:f.write(chunk);bar.update(len(chunk))
            if expected and part.stat().st_size!=expected:
                raise IOError(f'Incomplete download {part.stat().st_size} != {expected}')
            part.replace(dest)
            return dest
        except (OSError,requests.RequestException) as err:
            print(f'Download attempt {attempt+1}/{retries}: {type(err).__name__}: {str(err)[:180]}')
            if attempt==retries-1:raise RuntimeError(f'Cannot download {url}. Enable Kaggle Internet, check free disk, and rerun. Partial file: {part}') from err
            time.sleep(min(2**attempt,15))

session=requests.Session()
session.headers.update({'User-Agent':'Research-ISIC-2019-audit/2.0 (academic non-commercial)'})
archive_path=download(session,URLS['images'],CACHE/'ISIC_2019_Training_Input.zip')
gt_path=download(session,URLS['ground_truth'],ROOT/'source'/'ISIC_2019_Training_GroundTruth.csv')
meta_path=download(session,URLS['metadata'],ROOT/'source'/'ISIC_2019_Training_Metadata.csv')
source_sha={'ground_truth_sha256':file_sha256(gt_path),'metadata_sha256':file_sha256(meta_path),
            'images_zip_sha256':file_sha256(archive_path)}
print('Source ZIP:',round(archive_path.stat().st_size/2**30,2),'GiB')
print('Files downloaded, source metadata hashes captured.')

ISIC_2019_Training_Input.zip:   0%|          | 0.00/9.77G [00:00<?, ?B/s]

ISIC_2019_Training_GroundTruth.csv:   0%|          | 0.00/1.29M [00:00<?, ?B/s]

ISIC_2019_Training_Metadata.csv:   0%|          | 0.00/1.21M [00:00<?, ?B/s]

Source ZIP: 9.1 GiB
Files downloaded, source metadata hashes captured.


## 2. Read official labels and metadata and audit the source archive's image IDs

CSV records are matched to ZIP members by their original ISIC image IDs. The full input
metadata, all label records, and their unique-image counts remain traceable in the export.

In [3]:
def id_column(frame):
    for c in ['image','isic_id','image_id','image_name','file_name']:
        if c in frame.columns:return c
    raise ValueError('Missing image identifier column: '+str(frame.columns.tolist()))

def normalized_id(series):
    v=series.astype('string').str.strip()
    return v.mask(v.str.casefold().isin({'','nan','na','none','null','unknown','unk','n/a','0'}))

def read_id_table(path):
    frame=pd.read_csv(path,low_memory=False)
    frame.columns=frame.columns.str.strip()
    frame['image_id']=normalized_id(frame[id_column(frame)]).str.replace(r'\.(jpg|jpeg|png|webp)$','',regex=True,flags=re.I)
    if frame.image_id.isna().any():raise ValueError(f'Blank image ID in {path}')
    if frame.image_id.duplicated().any():raise ValueError(f'Duplicate image IDs in {path}')
    return frame

gt=read_id_table(gt_path);metadata=read_id_table(meta_path)
if not set(CLASSES).issubset(gt.columns):raise ValueError(f'Official ground truth class columns missing: {set(CLASSES)-set(gt.columns)}')
values=gt[CLASSES].apply(pd.to_numeric,errors='coerce')
gt_valid=values.isin([0,1]).all(axis=1)&values.sum(axis=1).eq(1)
gt['label']=pd.Series(pd.NA,index=gt.index,dtype='string')
gt.loc[gt_valid,'label']=values.loc[gt_valid].idxmax(axis=1)
gt['gt_valid']=gt_valid
metadata=metadata.rename(columns={'lesion_id':'lesion_id_official','patient_id':'patient_id_official'})
for col in ['lesion_id_official','patient_id_official','age_approx','sex','anatom_site_general']:
    if col not in metadata:metadata[col]=pd.NA
metadata=metadata[['image_id','lesion_id_official','patient_id_official','age_approx','sex','anatom_site_general']]
registry=gt[['image_id','label','gt_valid']].merge(metadata,on='image_id',how='outer',indicator='source_csv_presence',validate='one_to_one')
registry['lesion_id']=normalized_id(registry.lesion_id_official)
registry['patient_id']=normalized_id(registry.patient_id_official)
print('Ground-truth rows:',len(gt),'Metadata rows:',len(metadata))

with zipfile.ZipFile(archive_path) as zf:
    members=[i for i in zf.infolist() if not i.is_dir() and Path(i.filename).suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
    zip_idx=pd.DataFrame({'image_id':[Path(m.filename).stem for m in members],
                          'source_member':[m.filename for m in members],
                          'source_uncompressed_bytes':[m.file_size for m in members]})
    if zip_idx.image_id.duplicated().any():
        raise RuntimeError('Duplicate image IDs within official archive. Resolve source ambiguity before continuing.')
    if len(members)!=EXPECTED:raise RuntimeError(f'ISIC ZIP count is {len(members)}; expected {EXPECTED}. Verify URL and archive.')
    if len(gt)!=EXPECTED or len(metadata)!=EXPECTED:
        raise RuntimeError(f'Incomplete metadata: GT={len(gt)}, metadata={len(metadata)}, expected={EXPECTED}')
    missing_from_zip=set(gt.image_id)-set(zip_idx.image_id)
    if missing_from_zip:raise RuntimeError(f'{len(missing_from_zip)} ground-truth images missing from ZIP')

registry=registry.merge(zip_idx,on='image_id',how='outer',validate='one_to_one')
registry=registry.sort_values('image_id').reset_index(drop=True)
print('Source ZIP images:',len(zip_idx),'Registry entries:',len(registry))
print('Missing lesion IDs:',int(registry.lesion_id.isna().sum()),'Missing patient IDs:',int(registry.patient_id.isna().sum()))

Ground-truth rows: 25331 Metadata rows: 25331
Source ZIP images: 25331 Registry entries: 25331
Missing lesion IDs: 2084 Missing patient IDs: 25331


## 3. Image-by-image source audit and persistent 224×224 export

Every official image is decoded directly from the ZIP. We save source byte-level SHA256 and
dimensional/perceptual fingerprints plus a reusable 224×224 JPEG under `images/`.
**The downstream dataset uses a relative `image_relpath`, not a temporary Kaggle path.**
This enables adding this notebook's output as the only input for all later notebooks.

In [4]:
def hashes(rgb):
    grey=rgb.convert('L')
    pix=np.asarray(grey.resize((32,32),Image.Resampling.LANCZOS),dtype=np.float32)
    block=dctn(pix,type=2,norm='ortho')[:8,:8]
    bits=block>(np.median(block.flatten()[1:]))
    ph=f'{int.from_bytes(np.packbits(bits).tobytes(),"big"):016x}'
    arr=np.asarray(grey.resize((9,8),Image.Resampling.LANCZOS),dtype=np.uint8)
    dbits=arr[:,:-1]>arr[:,1:]
    dh=f'{int.from_bytes(np.packbits(dbits.flatten()).tobytes(),"big"):016x}'
    return ph,dh

def resize_pad(rgb,size):
    edges_small=np.asarray(rgb.resize((32,32),Image.Resampling.BILINEAR))
    edges=np.concatenate([edges_small[0],edges_small[-1],edges_small[:,0],edges_small[:,-1]],axis=0)
    color=tuple(np.median(edges,axis=0).astype(np.uint8).tolist())
    return ImageOps.pad(rgb,(size,size),method=Image.Resampling.BICUBIC,color=color)

def audit_member(zf,member):
    image_id=Path(member.filename).stem
    rel=f'images/{image_id}.jpg'
    dest=ROOT/rel
    try:
        with zf.open(member) as f:data=f.read()  # ZIP validates CRC on full member read
        sha=hashlib.sha256(data).hexdigest()
        import io
        with Image.open(io.BytesIO(data)) as source:
            source.load()
            rgb=ImageOps.exif_transpose(source).convert('RGB')
        w,h=rgb.size
        ph,dh=hashes(rgb)
        processed=resize_pad(rgb,SIZE)
        processed.save(dest,format='JPEG',quality=JPEG_QUALITY,subsampling=0,optimize=True)
        with Image.open(dest) as check:
            check.verify()
        return {'image_id':image_id,'width':w,'height':h,'original_bytes':len(data),
                'source_sha256':sha,'phash':ph,'dhash':dh,'image_relpath':rel,
                'resized_bytes':dest.stat().st_size,'resized_sha256':file_sha256(dest),
                'image_error':''}
    except Exception as exc:
        if dest.exists():dest.unlink()
        return {'image_id':image_id,'width':np.nan,'height':np.nan,'original_bytes':member.file_size,
                'source_sha256':pd.NA,'phash':pd.NA,'dhash':pd.NA,'image_relpath':pd.NA,
                'resized_bytes':0,'resized_sha256':pd.NA,'image_error':f'{type(exc).__name__}: {str(exc)[:200]}'}

# Scan progress is checkpointed every 250 images. Reuse only against the same source ZIP
# and unchanged preprocessing configuration; verify cached bytes before accepting a row.
scan_csv=ROOT/'audit'/'image_scan_progress.csv'
scan_state=ROOT/'audit'/'image_scan_state.json'
scan_key={'scan_version':SCAN_VERSION,'images_zip_sha256':source_sha['images_zip_sha256'],
          'image_size':SIZE,'jpeg_quality':JPEG_QUALITY,'expected_images':EXPECTED}
previous={}
if scan_csv.exists() and scan_state.exists():
    try:
        if json.loads(scan_state.read_text())==scan_key:
            prev=pd.read_csv(scan_csv,dtype={'image_id':str,'source_sha256':'string',
                'phash':'string','dhash':'string','image_relpath':'string','resized_sha256':'string'})
            prev=prev.drop_duplicates('image_id',keep='last')
            for r in tqdm(prev.to_dict('records'),desc='Verify cached images',unit='record'):
                rel=r.get('image_relpath')
                if not isinstance(rel,str) or not rel:continue
                dest=ROOT/rel
                if dest.is_file() and dest.stat().st_size==int(r['resized_bytes']) and file_sha256(dest)==r['resized_sha256']:
                    previous[r['image_id']]=r
            print('Verified previously exported images:',len(previous))
        else:print('Scan input or settings changed; will rescan from source ZIP.')
    except Exception as exc:
        print('Existing scan record unusable; rescan:',type(exc).__name__,str(exc)[:120]);previous={}
results=[]
scan_state.write_text(json.dumps(scan_key,indent=2))
with zipfile.ZipFile(archive_path) as zf:
    for n,m in enumerate(tqdm(members,desc='Decode, audit, export images',unit='image'),1):
        image_id=Path(m.filename).stem
        results.append(previous[image_id] if image_id in previous else audit_member(zf,m))
        if n%250==0:
            pd.DataFrame(results).to_csv(scan_csv,index=False)
    pd.DataFrame(results).to_csv(scan_csv,index=False)
scanned=pd.DataFrame(results)
registry=registry.merge(scanned,on='image_id',how='left',validate='one_to_one')
registry['class_index']=registry.label.map(CLASS_TO_ID)
registry['status_reason']=''
def reason(mask,name):
    mask=pd.Series(mask,index=registry.index).fillna(False).astype(bool)
    registry.loc[mask,'status_reason']=registry.loc[mask,'status_reason'].map(lambda old:';'.join(filter(None,[old,name])))
reason(registry.source_member.isna(),'missing_source_image')
reason(~registry.gt_valid.fillna(False).astype(bool),'missing_or_invalid_label')
reason(registry.image_error.fillna('').ne(''),'image_decode_or_export_error')
registry['sha_duplicate_count']=registry.groupby('source_sha256',dropna=True).image_id.transform('size').fillna(0).astype(int)
registry['lesion_image_count']=registry.groupby('lesion_id',dropna=True).image_id.transform('size').fillna(0).astype(int)
registry['patient_image_count']=registry.groupby('patient_id',dropna=True).image_id.transform('size').fillna(0).astype(int)
registry.to_csv(ROOT/'audit'/'master_before_grouping.csv',index=False)
print('Saved 224px images:',int(registry.image_relpath.notna().sum()),'of',EXPECTED)
print('Original SHA256 duplicates, surplus images:',int(registry.source_sha256.dropna().duplicated().sum()))

Decode, audit, export images:   0%|          | 0/25331 [00:00<?, ?image/s]

Saved 224px images: 25331 of 25331
Original SHA256 duplicates, surplus images: 50


## 4. Near-duplicate candidates, identity conflict checks and verified grouping

pHash/dHash pairs are NOT verified lesion matches. They are kept as review evidence only.
Group by official lesion/patient identifiers and exact source bytes; do not merge visual
candidate pairs into group IDs. When a possible cross-group visual match exists, both
identity groups become `train_only`: this prevents candidate overlap with held-out sets
without transitive giant clusters. Unresolved cases are recorded for professor review.

In [5]:
class BKNode:
    def __init__(self,value,pos):self.value=value;self.pos=[pos];self.children={}
class BKTree:
    def __init__(self):self.root=None
    def add(self,value,pos):
        if self.root is None:self.root=BKNode(value,pos);return
        node=self.root
        while True:
            d=(value^node.value).bit_count()
            if d==0:node.pos.append(pos);return
            if d not in node.children:node.children[d]=BKNode(value,pos);return
            node=node.children[d]
    def query(self,value,radius):
        if self.root is None:return []
        result=[];todo=[self.root]
        while todo:
            node=todo.pop();d=(value^node.value).bit_count()
            if d<=radius:result.append((d,node.pos))
            todo.extend(child for dist,child in node.children.items() if d-radius<=dist<=d+radius)
        return result

near=[];tree=BKTree()
valid_idx=registry.index[registry.phash.notna()&registry.dhash.notna()&registry.status_reason.eq('')].tolist()
# Cache scalar values: pandas .loc inside a large all-pairs loop is very slow.
image_ids=registry.image_id.tolist()
labels=registry.label.tolist()
lesion_ids=registry.lesion_id.tolist()
source_hashes=registry.source_sha256.tolist()
phash_int={i:int(registry.at[i,'phash'],16) for i in valid_idx}
dhash_int={i:int(registry.at[i,'dhash'],16) for i in valid_idx}
for idx in tqdm(valid_idx,desc='Find near-duplicate candidates'):
    p=phash_int[idx];d=dhash_int[idx]
    for ph_dist,others in tree.query(p,NEAR_PHASH_RADIUS):
        for j in others:
            dh_dist=(d^dhash_int[j]).bit_count()
            if dh_dist<=NEAR_DHASH_RADIUS and source_hashes[idx]!=source_hashes[j]:
                near.append({'image_id_a':image_ids[j],'image_id_b':image_ids[idx],
                             'phash_distance':ph_dist,'dhash_distance':dh_dist,
                             'label_a':labels[j],'label_b':labels[idx],
                             'lesion_id_a':lesion_ids[j],'lesion_id_b':lesion_ids[idx]})
                if len(near)>MAX_NEAR_PAIRS:
                    pd.DataFrame(near).to_csv(ROOT/'audit'/'near_duplicate_pairs_partial.csv',index=False)
                    raise RuntimeError('Excessive near-duplicate pairs; inspect partial review file.')
    tree.add(p,idx)
near_cols=['image_id_a','image_id_b','phash_distance','dhash_distance',
           'label_a','label_b','lesion_id_a','lesion_id_b']
near_df=pd.DataFrame(near,columns=near_cols)

class DSU:
    def __init__(self,n):self.p=list(range(n));self.rank=[0]*n
    def find(self,x):
        while x!=self.p[x]:self.p[x]=self.p[self.p[x]];x=self.p[x]
        return x
    def join(self,a,b):
        a=self.find(a);b=self.find(b)
        if a==b:return
        if self.rank[a]<self.rank[b]:a,b=b,a
        self.p[b]=a
        if self.rank[a]==self.rank[b]:self.rank[a]+=1

# ONLY confirmed metadata relationships and exact-byte identity may union groups.
dsu=DSU(len(registry))
for col in ['lesion_id','patient_id','source_sha256']:
    for _,idx in registry.loc[registry[col].notna()].groupby(col).groups.items():
        ids=list(idx)
        for j in ids[1:]:dsu.join(ids[0],j)
groups=defaultdict(list)
for idx in range(len(registry)):groups[dsu.find(idx)].append(idx)
name_by_root={root:hashlib.sha256('|'.join(sorted(registry.loc[indices,'image_id'].tolist())).encode()).hexdigest()[:16]
              for root,indices in groups.items()}
registry['group_id']=[name_by_root[dsu.find(i)] for i in range(len(registry))]

# Do not invent lesion identities for records with missing official metadata.
for col in ['lesion_id','source_sha256']:
    present=registry.loc[registry[col].notna()&registry.label.notna()]
    bad=set(present.groupby(col).label.nunique().loc[lambda v:v>1].index)
    reason(registry[col].isin(bad),f'conflicting_labels_within_{col}')
for col in ['lesion_id','patient_id']:
    bad=set(registry.loc[registry.source_sha256.notna()].groupby('source_sha256')[col].nunique(dropna=True).loc[lambda v:v>1].index)
    reason(registry.source_sha256.isin(bad),f'conflicting_{col}_for_same_bytes')
reason(registry.lesion_id.isna()&registry.patient_id.isna(),'identity_not_verified')
ordered=registry.loc[registry.source_sha256.notna()].sort_values(['source_sha256','image_id'])
extra_ids=set(ordered.loc[ordered.duplicated('source_sha256',keep='first'),'image_id'])
reason(registry.image_id.isin(extra_ids),'redundant_exact_duplicate')
conflicted_groups=set(registry.loc[registry.status_reason.str.contains('conflicting_',regex=False),'group_id'])
reason(registry.group_id.isin(conflicted_groups),'connected_to_identity_or_label_conflict')

registry['eligible']=registry.status_reason.eq('')
registry['split']='quarantine'
idx_by_id={v:i for i,v in enumerate(registry.image_id.tolist())}
if len(near_df):
    ra=registry.loc[[idx_by_id[k] for k in near_df.image_id_a]].reset_index(drop=True)
    rb=registry.loc[[idx_by_id[k] for k in near_df.image_id_b]].reset_index(drop=True)
    near_df['group_id_a']=ra.group_id.to_numpy()
    near_df['group_id_b']=rb.group_id.to_numpy()
    near_df['eligible_a']=ra.eligible.to_numpy()
    near_df['eligible_b']=rb.eligible.to_numpy()
    near_df['same_verified_group']=near_df.group_id_a.eq(near_df.group_id_b)
    near_df['unresolved_cross_group']=~near_df.same_verified_group
    near_df['label_disagreement']=near_df.label_a.ne(near_df.label_b)
    near_df['review_status']='unreviewed_candidate'
else:
    for c in ['group_id_a','group_id_b','eligible_a','eligible_b','same_verified_group',
              'unresolved_cross_group','label_disagreement','review_status']:
        near_df[c]=pd.Series(dtype='object')
# A suspected link is never upgraded to a *verified* same-lesion identity.
# Both eligible endpoint groups are TRAIN-ONLY, not merged; this avoids any candidate pair
# between train, meta_train, validation and test. The unsupervised SSL pool is train only.
review_pairs=near_df.loc[near_df.unresolved_cross_group.astype(bool)&near_df.eligible_a.astype(bool)&near_df.eligible_b.astype(bool)]
protected_groups=set(review_pairs.group_id_a).union(review_pairs.group_id_b)
registry['train_only_candidate_group']=registry.group_id.isin(protected_groups)
near_df.to_csv(ROOT/'audit'/'near_duplicate_candidates.csv',index=False)
registry.to_csv(ROOT/'audit'/'identity_master_registry.csv',index=False)
exclusion=registry.loc[~registry.eligible,'status_reason'].str.split(';').explode().value_counts().rename_axis('reason').reset_index(name='image_count')
exclusion.to_csv(ROOT/'audit'/'exclusion_reasons.csv',index=False)
print('Candidate near-duplicate pairs:',len(near_df))
print('Unresolved eligible cross-group candidate pairs:',len(review_pairs))
print('Verified groups reserved for train:',len(protected_groups))
print('Train-only eligible images:',int((registry.eligible&registry.train_only_candidate_group).sum()))
print('Known lesion IDs:',registry.lesion_id.nunique(dropna=True))
print('Eligible:',int(registry.eligible.sum()),'Quarantined:',int((~registry.eligible).sum()))
if not registry.eligible.any():
    raise RuntimeError('No eligible images with verified identity; audit saved. Cannot construct strict split.')
max_group=int(registry.loc[registry.eligible,'group_id'].value_counts().max())
print('Largest VERIFIED eligible identity group:',max_group)
if max_group>max(100,int(.1*registry.eligible.sum())):
    raise RuntimeError('Large group from VERIFIED metadata/exact hashes: inspect official IDs and audit before proceeding.')

Find near-duplicate candidates:   0%|          | 0/25331 [00:00<?, ?it/s]

Candidate near-duplicate pairs: 2717
Unresolved eligible cross-group candidate pairs: 1929
Verified groups reserved for train: 901
Train-only eligible images: 3373
Known lesion IDs: 11847
Eligible: 23187 Quarantined: 2144
Largest VERIFIED eligible identity group: 31


## 5. Freeze strict train / meta-train / validation / test partitions

Identity groups are based on OFFICIAL lesion IDs, available patient IDs and exact bytes.
Train-only candidate groups cannot enter meta-train/validation/test. The remaining clean
groups are stratified across four partitions; report any deviation from 70/10/10/10.
Never choose partitions based on any model outcomes.

In [6]:
work=registry.loc[registry.eligible].copy()
work['class_index']=work.class_index.astype(int)
safe=work.loc[~work.train_only_candidate_group].copy()
fixed=work.loc[work.train_only_candidate_group].copy()
print('Holdout-eligible clean images:',len(safe),'Protected train-only images:',len(fixed))
print('Holdout-eligible groups by class:',safe.groupby('class_index').group_id.nunique().reindex(range(8),fill_value=0).to_dict())
if safe.empty:raise RuntimeError('No candidate-free groups remain for held-out evaluation; review candidates first.')
clean_class_groups=safe.groupby('class_index').group_id.nunique().reindex(range(8),fill_value=0)
if (clean_class_groups<4).any():
    raise RuntimeError('Too few candidate-free verified groups for holdouts; inspect review pairs and metadata. '+str(clean_class_groups.to_dict()))
all_class_total=work.class_index.value_counts().reindex(range(8),fill_value=0).to_numpy()
all_freq=all_class_total/all_class_total.sum()
clean_fraction=len(safe)/len(work)
ideal_folds=max(4,round(clean_fraction/.10))
fold_options=sorted({k for k in range(max(4,ideal_folds-2),min(14,ideal_folds+3)) if k<=safe.group_id.nunique()})
print('Grouped split fold candidates:',fold_options)
best=None
for nf in fold_options:
    for attempt in range(12):
        cv=StratifiedGroupKFold(n_splits=nf,shuffle=True,random_state=SEED+100*nf+attempt)
        fold=np.full(len(safe),-1,dtype=np.int16)
        for k,(_,indices) in enumerate(cv.split(safe,safe.class_index,groups=safe.group_id)):
            fold[indices]=k
        if (fold<0).any():raise AssertionError('Unassigned grouped folds')
        chooser=np.random.default_rng(SEED+100*nf+attempt)
        held=chooser.choice(nf,size=3,replace=False)
        mapping=np.array(['train']*nf,dtype=object)
        mapping[held]=['test','validation','meta_train']
        split=mapping[fold]
        split_series=pd.Series(split,index=safe.index)
        all_assign=pd.concat([split_series,pd.Series('train',index=fixed.index)]).rename('split')
        ct=pd.crosstab(all_assign,work.class_index).reindex(index=['train','meta_train','validation','test'],columns=range(8),fill_value=0)
        if (ct==0).any().any():continue
        actual=ct.sum(axis=1).to_numpy()
        target=np.array([.7,.1,.1,.1])*len(work)
        size_error=float(np.mean(np.abs(actual-target)/len(work)))
        distributions=ct.div(ct.sum(axis=1),axis=0).to_numpy()
        dist_error=float(np.mean(np.abs(distributions-all_freq)))
        class_holdout_error=float(np.mean(np.abs(ct.loc[['meta_train','validation','test']].to_numpy()/np.maximum(all_class_total,1)[None,:]-.1)))
        score=4*size_error+dist_error+class_holdout_error
        if best is None or score<best[0]:best=(score,split_series,SEED+100*nf+attempt,nf)
if best is None:
    raise RuntimeError('Cannot create all eight classes in three independent holdouts from candidate-free groups. Review near-duplicate flags; never fall back to an image-level split.')
registry.loc[fixed.index,'split']='train'
registry.loc[best[1].index,'split']=best[1]

class_table=pd.crosstab(registry.split,registry.label).reindex(index=['train','meta_train','validation','test','quarantine'],columns=CLASSES,fill_value=0)
class_table['total']=class_table.sum(axis=1)
class_table.to_csv(ROOT/'splits'/'class_distribution.csv')
display(class_table)
for col in ['group_id','lesion_id','patient_id','source_sha256']:
    df=registry.loc[registry.split.ne('quarantine')&registry[col].notna(),[col,'split']].drop_duplicates()
    overlaps=int(df.groupby(col).split.nunique().gt(1).sum())
    print(col,'cross-partition overlaps:',overlaps)
    assert overlaps==0
assert registry.loc[registry.train_only_candidate_group&registry.eligible,'split'].eq('train').all()
if len(near_df):
    after=registry.set_index('image_id')['split']
    near_df['split_a']=near_df.image_id_a.map(after)
    near_df['split_b']=near_df.image_id_b.map(after)
    near_df['cross_split']=near_df.split_a.ne(near_df.split_b)
    near_df['cross_active_split']=near_df.cross_split&near_df.split_a.ne('quarantine')&near_df.split_b.ne('quarantine')
    near_df.to_csv(ROOT/'audit'/'near_duplicate_candidates.csv',index=False)
    unresolved=int(near_df.cross_active_split.sum())
    print('Unresolved near-duplicate candidate pairs crossing ACTIVE splits:',unresolved)
    assert unresolved==0,'Visual review pairs must not cross active partitions'
for split in ['train','meta_train','validation','test','quarantine']:
    registry.loc[registry.split.eq(split)].to_csv(ROOT/'splits'/f'{split}_images.csv',index=False)
registry.to_csv(ROOT/'splits'/'grouped_splits_all_images.csv',index=False)
# Each verified group is traceable independently of the near-duplicate review graph.
group_summary=registry.groupby('group_id',sort=True).agg(
    image_count=('image_id','size'),
    verified_lesion_ids=('lesion_id','nunique'),
    verified_patient_ids=('patient_id','nunique'),
    distinct_labels=('label','nunique'),
    eligible_images=('eligible','sum'),
    reserved_train_only=('train_only_candidate_group','max'),
    split=('split',lambda v:','.join(sorted(v.unique()))))
group_summary.to_csv(ROOT/'audit'/'verified_identity_groups.csv')
review_summary={'all_near_candidate_pairs':int(len(near_df)),
    'unresolved_cross_group_pairs':int(near_df.unresolved_cross_group.sum()) if len(near_df) else 0,
    'unresolved_eligible_cross_group_pairs':int(len(review_pairs)),
    'candidate_pairs_with_different_labels':int(near_df.label_disagreement.sum()) if len(near_df) else 0,
    'candidate_train_only_groups':len(protected_groups),
    'candidate_train_only_eligible_images':int((registry.eligible&registry.train_only_candidate_group).sum()),
    'candidate_pairs_crossing_active_splits':int(near_df.cross_active_split.sum()) if len(near_df) else 0,
    'review_status':'UNREVIEWED: pHash/dHash similarity is not a verified same-lesion diagnosis'}
(ROOT/'audit'/'near_duplicate_review_summary.json').write_text(json.dumps(review_summary,indent=2))
assert registry.eligible.sum()==registry.split.ne('quarantine').sum()
print('Chosen grouped split seed:',best[2],'number of folds:',best[3])
print('Achieved train/meta/validation/test percentages:',
      (100*registry.split.value_counts().reindex(['train','meta_train','validation','test'],fill_value=0)/len(work)).round(2).to_dict())

Holdout-eligible clean images: 19814 Protected train-only images: 3373
Holdout-eligible groups by class: {0: 1191, 1: 6738, 2: 1106, 3: 255, 4: 1185, 5: 102, 6: 124, 7: 238}
Grouped split fold candidates: [7, 8, 9, 10, 11]


label        MEL    NV   BCC   AK   BKL   DF  VASC  SCC  total
split                                                         
train       3079  7955  2352  625  1667  178   157  477  16490
meta_train   365  1095   342   91   274   26    33   41   2267
validation   328  1156   289   73   240   10    41   49   2186
test         403  1102   333   70   228   25    22   61   2244
quarantine   347  1567     7    8   215    0     0    0   2144

group_id cross-partition overlaps: 0
lesion_id cross-partition overlaps: 0
patient_id cross-partition overlaps: 0
source_sha256 cross-partition overlaps: 0
Unresolved near-duplicate candidate pairs crossing ACTIVE splits: 0
Chosen grouped split seed: 948 number of folds: 9
Achieved train/meta/validation/test percentages: {'train': 71.12, 'meta_train': 9.78, 'validation': 9.43, 'test': 9.68}


## 6. Nested label budgets; all-image budget tracking

Budgets apply only to the ordinary `train` partition. Entire train groups are selected.
ALL permitted train images (not validation, meta-train or test) may be used **without labels**
for SSL pretraining. This creates fair nested low-label comparisons.

In [7]:
train=registry.loc[registry.split.eq('train')].copy()
groups_list=[]
for gid,sub in train.groupby('group_id',sort=True):
    groups_list.append((gid,sub.class_index.astype(int).value_counts().reindex(range(8),fill_value=0).to_numpy(dtype=int)))
rng=np.random.default_rng(SEED)
rng.shuffle(groups_list)
gids=np.array([x[0] for x in groups_list])
gmat=np.stack([x[1] for x in groups_list])
class_total=gmat.sum(axis=0)
selected=np.zeros(len(gids),dtype=bool)
current=np.zeros(8,dtype=int)
budget_groups={}
for pct in [5,10,20,50,100]:
    target=np.ceil(class_total*pct/100).astype(int)
    total_needed=math.ceil(len(train)*pct/100)
    if pct==100:selected[:]=True;current=class_total.copy()
    else:
        while int(current.sum())<total_needed or np.any(current==0):
            options=np.flatnonzero(~selected)
            if not len(options):break
            if len(options)>512:options=rng.choice(options,512,replace=False)
            updated=current[None,:]+gmat[options]
            missing=(updated==0).sum(axis=1)
            cost=np.mean(((updated-target[None,:])/np.maximum(target,2)[None,:])**2,axis=1)
            cost+=.04*((updated.sum(axis=1)-total_needed)/max(total_needed,1))**2
            cost+=10*missing
            index=int(options[np.argmin(cost)])
            selected[index]=True;current+=gmat[index]
    chosen=set(gids[selected].tolist())
    subset=train.loc[train.group_id.isin(chosen)]
    if set(subset.label.unique())!=set(CLASSES):raise RuntimeError(f'Label budget {pct}% lacks one or more classes')
    budget_groups[pct]=chosen
    cols=['image_id','image_relpath','group_id','label','class_index','lesion_id','patient_id','split']
    subset[cols].to_csv(ROOT/'budgets'/f'train_labeled_{pct:03d}pct.csv',index=False)
    label_ct=subset.label.value_counts().reindex(CLASSES,fill_value=0)
    pd.DataFrame({'class':CLASSES,'n':label_ct.values}).to_csv(ROOT/'budgets'/f'budget_{pct:03d}_class_counts.csv',index=False)
    weights=np.sqrt(len(subset)/np.maximum(label_ct.to_numpy(),1));weights/=weights.mean()
    (ROOT/'budgets'/f'example_class_weights_{pct:03d}pct.json').write_text(json.dumps(dict(zip(CLASSES,weights.round(7).tolist())),indent=2))
    print(f'{pct:3d}%: {len(subset):6d} images, {len(chosen):5d} groups; class counts:',label_ct.to_dict())
for small,big in zip([5,10,20,50],[10,20,50,100]):assert budget_groups[small].issubset(budget_groups[big])
per_budget=[]
for pct,gids_selected in budget_groups.items():
    tmp=registry[['image_id','group_id','split','label','image_relpath']].copy()
    tmp['budget_pct']=pct
    tmp['label_visible_for_training']=tmp.split.eq('train')&tmp.group_id.isin(gids_selected)
    tmp['ssl_unlabeled_allowed']=tmp.split.eq('train')
    per_budget.append(tmp)
all_budget=pd.concat(per_budget,ignore_index=True)
all_budget.to_csv(ROOT/'budgets'/'all_image_label_budget_membership.csv',index=False)
assert len(all_budget)==len(registry)*5
assert not all_budget.loc[all_budget.split.ne('train'),'ssl_unlabeled_allowed'].any()

  5%:    828 images,    81 groups; class counts: {'MEL': 154, 'NV': 398, 'BCC': 118, 'AK': 33, 'BKL': 84, 'DF': 9, 'VASC': 8, 'SCC': 24}
 10%:   1651 images,   188 groups; class counts: {'MEL': 308, 'NV': 795, 'BCC': 236, 'AK': 63, 'BKL': 167, 'DF': 18, 'VASC': 16, 'SCC': 48}
 20%:   3300 images,   473 groups; class counts: {'MEL': 616, 'NV': 1588, 'BCC': 471, 'AK': 125, 'BKL': 334, 'DF': 38, 'VASC': 32, 'SCC': 96}
 50%:   8246 images,  2003 groups; class counts: {'MEL': 1540, 'NV': 3976, 'BCC': 1176, 'AK': 313, 'BKL': 834, 'DF': 89, 'VASC': 79, 'SCC': 239}
100%:  16490 images,  8193 groups; class counts: {'MEL': 3079, 'NV': 7955, 'BCC': 2352, 'AK': 625, 'BKL': 1667, 'DF': 178, 'VASC': 157, 'SCC': 477}


## 7. Visual, scientific and audit reports

Figures show recorded class frequencies and samples. Files include every image's original
hashes/dimensions and its reusable output path; figures are representative, not a substitute
for the complete CSV registry.

In [8]:
def show_saved(name):
    path=ROOT/'figures'/name
    if path.exists():display(NotebookImage(filename=str(path)))

coverage=pd.DataFrame([{'field':c,'available':int(registry[c].notna().sum()),'missing':int(registry[c].isna().sum()),
                       'coverage_pct':round(100*registry[c].notna().mean(),2)}
                       for c in ['label','lesion_id','patient_id','age_approx','sex','anatom_site_general']])
coverage.to_csv(ROOT/'audit'/'metadata_coverage.csv',index=False)
display(coverage)
fig,ax=plt.subplots(figsize=(8,4))
count_plot=registry.split.value_counts().reindex(['train','meta_train','validation','test','quarantine'],fill_value=0)
bars=ax.bar(count_plot.index,count_plot.values)
ax.bar_label(bars,padding=3)
ax.set(title='All source records accounted for (including quarantined)',ylabel='Number of source images')
fig.tight_layout();fig.savefig(ROOT/'figures'/'image_accounting_by_split.png',dpi=170);plt.close(fig)
show_saved('image_accounting_by_split.png')

ax=class_table.loc[['train','meta_train','validation','test'],CLASSES].T.plot(kind='bar',figsize=(12,5),width=.85)
ax.set(xlabel='Diagnosis',ylabel='Images',title='Identity-aware dataset splits')
plt.xticks(rotation=0);plt.tight_layout();plt.savefig(ROOT/'figures'/'split_class_distribution.png',dpi=170);plt.close()
show_saved('split_class_distribution.png')

fig,axes=plt.subplots(2,4,figsize=(12,6))
for ax,cls in zip(axes.flat,CLASSES):
    sub=registry[(registry.split=='train')&(registry.label==cls)].sort_values('image_id')
    if len(sub):
        with Image.open(ROOT/sub.iloc[0].image_relpath) as im:ax.imshow(im)
        ax.set_title(f'{cls} | {sub.iloc[0].image_id}',fontsize=10)
    ax.axis('off')
fig.suptitle('Representative reusable 224×224 training images');fig.tight_layout()
fig.savefig(ROOT/'figures'/'preprocessed_examples_by_class.png',dpi=155);plt.close(fig)
show_saved('preprocessed_examples_by_class.png')

import io
fig,axes=plt.subplots(8,2,figsize=(7,23))
with zipfile.ZipFile(archive_path) as zf:
    for row,cls in enumerate(CLASSES):
        source_row=registry[(registry.split=='train')&(registry.label==cls)].sort_values('image_id').iloc[0]
        with zf.open(source_row.source_member) as f:
            with Image.open(io.BytesIO(f.read())) as src:
                original=ImageOps.exif_transpose(src).convert('RGB')
        with Image.open(ROOT/source_row.image_relpath) as img:processed=img.copy()
        axes[row,0].imshow(original);axes[row,1].imshow(processed)
        axes[row,0].set_title(f'{cls} — Original ({original.width}×{original.height})',fontsize=10)
        axes[row,1].set_title(f'{cls} — Exported ({SIZE}×{SIZE})',fontsize=10)
        for axis in axes[row]:axis.axis('off')
fig.suptitle('Source versus aspect-preserving standardized image')
fig.tight_layout();fig.savefig(ROOT/'figures'/'source_vs_preprocessed.png',dpi=150);plt.close(fig)
show_saved('source_vs_preprocessed.png')

example=train.sort_values('image_id').iloc[0]
with Image.open(ROOT/example.image_relpath) as im:base=im.convert('RGB')
aug=[base,ImageOps.mirror(base),ImageOps.flip(base),base.rotate(90),base.rotate(180)]
fig,axes=plt.subplots(1,5,figsize=(12,3))
for ax,im,title in zip(axes,aug,['Base','Horizontal flip','Vertical flip','Rotate 90°','Rotate 180°']):
    ax.imshow(im);ax.axis('off');ax.set_title(title,fontsize=9)
fig.tight_layout();fig.savefig(ROOT/'figures'/'training_augmentation_examples.png',dpi=155);plt.close(fig)
show_saved('training_augmentation_examples.png')

fig,ax=plt.subplots(figsize=(7,4))
ax.bar([str(k)+'%' for k in budget_groups],[int(train.group_id.isin(v).sum()) for v in budget_groups.values()])
ax.set(title='Nested labeled training budgets',xlabel='Labeling budget',ylabel='Actual images')
fig.tight_layout();fig.savefig(ROOT/'figures'/'label_budget_counts.png',dpi=170);plt.close(fig)
show_saved('label_budget_counts.png')

if len(near_df):
    sample=near_df.head(6)
    fig,axes=plt.subplots(len(sample),2,figsize=(7,2.8*len(sample)),squeeze=False)
    for i,pair in enumerate(sample.itertuples(index=False)):
        for j,image_id in enumerate([pair.image_id_a,pair.image_id_b]):
            record=registry.loc[registry.image_id.eq(image_id)].iloc[0]
            with Image.open(ROOT/record.image_relpath) as im:axes[i,j].imshow(im)
            axes[i,j].axis('off');axes[i,j].set_title(f'{image_id} | pH={pair.phash_distance}, dH={pair.dhash_distance}',fontsize=9)
    fig.tight_layout();fig.savefig(ROOT/'figures'/'near_duplicate_review_examples.png',dpi=155);plt.close(fig)

preprocess_config={
 'geometry':'EXIF transpose, RGB conversion, aspect-preserving resize with median-edge RGB padding',
 'image_size':[SIZE,SIZE],'format':'JPEG','jpeg_quality':JPEG_QUALITY,
 'source':'Official ISIC 2019 training ZIP; originals audited, cached images exported',
 'normalization':'Backbone-specific normalization applied only in later training notebooks',
 'augmentation':'Apply random augmentations only in training; evaluation must be deterministic',
 'ssl_allowed_splits':['train'],'identity_missing_policy':'Quarantine from strict study',
 'near_duplicate_policy':f'Review only; pHash<={NEAR_PHASH_RADIUS}, dHash<={NEAR_DHASH_RADIUS}; eligible cross-group endpoints reserved for train, NEVER unioned',
 'identity_claim':'Group-disjoint for verified metadata/exact bytes; all unresolved cross-group visual candidate endpoints train only; patient independence not established'
}
(ROOT/'reports'/'preprocessing_config.json').write_text(json.dumps(preprocess_config,indent=2))

                 field  available  missing  coverage_pct
0                label      25331        0        100.00
1            lesion_id      23247     2084         91.77
2           patient_id          0    25331          0.00
3           age_approx      24894      437         98.27
4                  sex      24947      384         98.48
5  anatom_site_general      22700     2631         89.61

879

## 8. Freeze manifests and verify every exported image

Kaggle's notebook output contains the **entire resized image set** plus source/provenance tables.
Attach only this completed notebook output as a dataset input to Notebook 02 onward.
For example, find `reports/protocol.json` in `/kaggle/input/<notebook-output>/...` and
load image paths using `Path(dataset_root)/image_relpath`, NOT old `/kaggle/working` paths.

In [9]:
manifest_cols=['image_id','image_relpath','label','class_index','lesion_id','patient_id','group_id','split','train_only_candidate_group',
               'eligible','status_reason','width','height','original_bytes','source_sha256','phash','dhash',
               'resized_bytes','resized_sha256','source_member','age_approx','sex','anatom_site_general']
manifest=registry[manifest_cols].copy()
manifest.to_csv(ROOT/'splits'/'experiment_master_manifest.csv',index=False)
registry.to_csv(ROOT/'audit'/'complete_image_level_registry.csv',index=False)
accounting={
 'expected_source_images':EXPECTED,'source_zip_image_count':len(members),
 'ground_truth_rows':len(gt),'official_metadata_rows':len(metadata),
 'all_registered_images':len(registry),
 'exported_resized_images':int(registry.image_relpath.notna().sum()),
 'identity_verified_images':int((registry.lesion_id.notna()|registry.patient_id.notna()).sum()),
 'unknown_identity_images':int((registry.lesion_id.isna()&registry.patient_id.isna()).sum()),
 'unique_lesion_ids':int(registry.lesion_id.nunique()),
 'unique_patient_ids':int(registry.patient_id.nunique()),
 'images_from_repeated_lesions':int(registry.lesion_image_count.gt(1).sum()),
 'byte_identical_duplicate_surplus':int(registry.source_sha256.dropna().duplicated().sum()),
 'near_duplicate_candidate_pairs':len(near_df),
 'unresolved_eligible_candidate_pairs':len(review_pairs),
 'candidate_train_only_groups':len(protected_groups),
 'candidate_train_only_eligible_images':int((registry.eligible&registry.train_only_candidate_group).sum()),
 'cross_active_split_near_candidate_pairs':int(near_df.cross_active_split.sum()) if len(near_df) else 0,
 'eligible_images':int(registry.eligible.sum()),
 'quarantined_images':int((~registry.eligible).sum()),
 **{f'{split}_images':int(registry.split.eq(split).sum()) for split in ['train','meta_train','validation','test']},
 'exported_images_bytes':int(registry.resized_bytes.fillna(0).sum())
}
assert len(registry)==EXPECTED
assert len(manifest.image_id.unique())==EXPECTED
assert int(registry.eligible.sum())+int((~registry.eligible).sum())==EXPECTED
assert int(registry.split.ne('quarantine').sum())==int(registry.eligible.sum())
assert len(list((ROOT/'images').glob('*.jpg')))==int(registry.image_relpath.notna().sum())
assert set(manifest.loc[manifest.split.ne('quarantine'),'image_relpath'].dropna()).issubset(set('images/'+p.name for p in (ROOT/'images').glob('*.jpg')))
if accounting['exported_resized_images']!=EXPECTED:
    raise RuntimeError('Not all original images could be decoded and exported. See audit/complete_image_level_registry.csv; do not publish as a complete dataset.')
(ROOT/'reports'/'dataset_accounting.json').write_text(json.dumps(accounting,indent=2))
protocol={
 'project':'lesion-n01-dataset-audit-grouped-splits','pipeline_version':PIPELINE_VERSION,
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'source_urls':URLS,'source_sha256':source_sha,
 'seed':SEED,'classes':CLASSES,'expected_source_images':EXPECTED,
 'split_proportions_target':{'train':.7,'meta_train':.1,'validation':.1,'test':.1},
 'grouped_split_seed':best[2],'grouped_split_folds':best[3],'label_budgets':[5,10,20,50,100],
 'preprocessing':preprocess_config,
 'manifest_sha256':file_sha256(ROOT/'splits'/'experiment_master_manifest.csv'),
 'data_root_portability':'image_relpath is relative to the folder containing reports/protocol.json',
 'training_inputs':'Use the images and CSV manifests from THIS notebook output; no new dataset download',
 'limitations':['Patient IDs may be absent: cannot claim fully patient-disjoint data',
                'Visual duplicate detection is approximate; all unresolved eligible cross-group candidate endpoints are restricted to training and still require manual review',
                'Strict study quarantines images without verified lesion or patient identity',
                'Patient-level isolation cannot be verified without patient identifiers; identities missing from source are not imputed'],
 'versions':{'python':sys.version.split()[0],'numpy':np.__version__,'pandas':pd.__version__,
             'pillow':Image.__version__,'scikit_learn':__import__('sklearn').__version__}
}
(ROOT/'reports'/'protocol.json').write_text(json.dumps(protocol,indent=2))

# The archive is retained until the portable loader has been saved in the final cell.
print('\nDATASET ACCOUNTING')
print(pd.Series(accounting).to_string())
print('\nManifest SHA256:',protocol['manifest_sha256'])
print('Output dataset root:',ROOT)
print('Saved image volume (GiB):',round(accounting['exported_images_bytes']/2**30,3))
print('Data audit and integrity verification completed; writing the reusable loader next.')


DATASET ACCOUNTING
expected_source_images                         25331
source_zip_image_count                         25331
ground_truth_rows                              25331
official_metadata_rows                         25331
all_registered_images                          25331
exported_resized_images                        25331
identity_verified_images                       23247
unknown_identity_images                         2084
unique_lesion_ids                              11847
unique_patient_ids                                 0
images_from_repeated_lesions                   16459
byte_identical_duplicate_surplus                  50
near_duplicate_candidate_pairs                  2717
unresolved_eligible_candidate_pairs             1929
candidate_train_only_groups                      901
candidate_train_only_eligible_images            3373
cross_active_split_near_candidate_pairs            0
eligible_images                                23187
quarantined_images        

## 9. Save a loader for the next Kaggle notebook

In Notebook 02 add the **completed output of Notebook 01** under Kaggle Inputs.
No separate original ISIC dataset is needed: all 224px images and the split files are present.
The saved loader resolves paths relative to the published Kaggle input folder and verifies
the manifest checksum before returning any training rows.

In [10]:
LOADER_CODE='''from pathlib import Path
import hashlib,json
import pandas as pd

def load_n01(root,split=None,require_images=True):
    root=Path(root).resolve()
    protocol=json.loads((root/'reports'/'protocol.json').read_text())
    manifest_path=root/'splits'/'experiment_master_manifest.csv'
    digest=hashlib.sha256(manifest_path.read_bytes()).hexdigest()
    if digest!=protocol['manifest_sha256']:
        raise RuntimeError('N01 dataset manifest checksum mismatch')
    df=pd.read_csv(manifest_path)
    if split is not None:df=df.loc[df.split.eq(split)].copy()
    if require_images:
        for rel in df.image_relpath.dropna():
            if Path(rel).is_absolute() or '..' in Path(rel).parts:
                raise RuntimeError('Unsafe path in N01 dataset manifest')
        if not all((root/p).is_file() for p in df.image_relpath.dropna()):
            raise FileNotFoundError('One or more cached images are missing from N01 output')
    return df,protocol

def find_n01_input(kaggle_input='/kaggle/input'):
    candidates=list(Path(kaggle_input).rglob('reports/protocol.json'))
    candidates=[p.parent.parent for p in candidates if 'lesion-n01' in str(p.parent.parent)]
    if len(candidates)!=1:
        raise RuntimeError(f'Expected exactly one N01 output in Kaggle Inputs; found {len(candidates)}: {candidates[:5]}')
    return candidates[0]
'''
(ROOT/'reports'/'n01_loader.py').write_text(LOADER_CODE,encoding='utf-8')
print('Next-stage loader saved:',ROOT/'reports'/'n01_loader.py')
assert (ROOT/'reports'/'n01_loader.py').is_file()
keep=os.getenv('N01_KEEP_SOURCE_ZIP','0').strip()=='1'
if not keep and urlparse(URLS['images']).scheme in ('http','https'):
    archive_path.unlink()
    print('Removed original source ZIP after final verification to conserve storage.')
print('SUCCESS — Add this completed N01 notebook output as the only dataset input for Notebook 02.')

Next-stage loader saved: /kaggle/working/lesion-n01-dataset-audit-grouped-splits/reports/n01_loader.py
Removed original source ZIP after final verification to conserve storage.
SUCCESS — Add this completed N01 notebook output as the only dataset input for Notebook 02.
